# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 4: Samouwaga (self-attention) - serce transformera

### Materiały i inspiracje

Ten notebook opiera się przede wszystkim na materiałach Andreja Karpathy'ego:

- [Let's build GPT: from scratch, in code, spelled out (YouTube)](https://www.youtube.com/watch?v=kCc8FmEb1nY) - kluczowy wykład, z którego pochodzi większość intuicji i kolejność wprowadzania pojęć w tym zeszycie.
- [nanoGPT (GitHub)](https://github.com/karpathy/nanoGPT) - referencyjna, minimalna implementacja GPT.
- [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html) - pełny kurs, z którego pochodzi też "makemore" (na nim wzorowany jest Notebook 3).
- [Attention Is All You Need (Vaswani i in., 2017)](https://arxiv.org/abs/1706.03762) - oryginalna praca wprowadzająca architekturę transformera.

### O czym jest ten notebook?

W [Notebooku 3](3_pytorch_network.ipynb) zbudowaliśmy MLP (perceptron wielowarstwowy). Patrzył on na ostatnie 3 znaki, sklejał ich embeddingi w jeden długi wektor i przepuszczał przez warstwy liniowe.

Problem: jeśli chcemy patrzeć na 32 albo 256 znaków wstecz, ten "sklejony" wektor robi się ogromny, a sieć musi się od zera uczyć, że pozycje są ze sobą powiązane.

**Pomysł samouwagi (self-attention)**: zamiast sklejać znaki w jeden wektor, niech każdy znak "zapyta" pozostałe: *"które z was są dla mnie ważne?"* i niech zbierze z nich informację jako średnią ważoną. To jest serce transformera (i całego nowoczesnego AI).

W tym notebooku:
1. Pokażemy, że samouwaga to po prostu sprytna **średnia ważona** poprzednich tokenów.
2. Zbudujemy ją krok po kroku - od zwykłej średniej, przez maskę trójkątną, aż po Q/K/V (query, key, value).
3. Złożymy z tego mini-model **z jedną głową uwagi** i wytrenujemy go na *Panu Tadeuszu*.

W [Notebooku 5](5_mini_gpt.ipynb) na bazie tego zbudujemy już pełny mini-GPT z wieloma głowami i kilkoma warstwami.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

torch.manual_seed(42)

### 1. Dane: Pan Tadeusz znak po znaku

Zaczynamy tak samo jak w Notebooku 3 - wczytujemy tekst, budujemy słownik i tensor treningowy/testowy. Tylko zwiększamy `context_size` (w transformerach często nazywany `block_size`).

In [ ]:
with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

# Cały tekst jako tensor liczb
data = torch.tensor([char2id[c] for c in text], dtype=torch.long)

split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
test_data = data[split_idx:]

print(f"Rozmiar słownika: {vocab_size}")
print(f"Trening: {len(train_data)} znaków, Test: {len(test_data)} znaków")

Zamiast budować ogromny tensor wszystkich par (kontekst, cel) jak w Notebooku 3, będziemy **losować paczki (batche)** w trakcie treningu. To standardowe podejście dla większych modeli i dłuższych kontekstów.

In [ ]:
block_size = 16  # Ile znaków wstecz model widzi
batch_size = 64

def losuj_paczke(split: str = "train") -> tuple[torch.Tensor, torch.Tensor]:
    """Losuje paczkę (batch) sekwencji długości block_size oraz ich "następników"."""
    zrodlo = train_data if split == "train" else test_data
    # Losujemy batch_size pozycji startowych
    ix = torch.randint(0, len(zrodlo) - block_size - 1, (batch_size,))
    x = torch.stack([zrodlo[i:i+block_size] for i in ix])
    # y to to samo co x, ale przesunięte o 1 - kolejny znak po każdej pozycji
    y = torch.stack([zrodlo[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = losuj_paczke()
print(f"x kształt: {xb.shape}  (batch_size, block_size)")
print(f"y kształt: {yb.shape}")
print("\nPrzykład pierwszej sekwencji:")
print(f"x: '{''.join(id2char[i.item()] for i in xb[0])}'")
print(f"y: '{''.join(id2char[i.item()] for i in yb[0])}'")

Zauważ ciekawą rzecz: w jednej sekwencji długości 16 ukrytych jest 16 osobnych przykładów uczących! Pozycja 0 uczy: "widząc znak `x[0]`, przewidź `y[0]`". Pozycja 1 uczy: "widząc znaki `x[0..1]`, przewidź `y[1]`". I tak dalej. Dzięki temu jeden batch daje nam mnóstwo sygnału do nauki.

### 2. "Trick" Karpathy'ego: średnia ważona poprzednich tokenów

Wyobraźmy sobie najprostszy pomysł: każdy token na pozycji `t` powinien "wiedzieć" coś o wszystkich poprzednich tokenach (od 0 do `t`). Jak to ugryźć?

**Pomysł 1 (najgłupszy, ale działa)**: każdy token bierze średnią arytmetyczną embeddingów wszystkich poprzednich tokenów.

Pokażemy to na małym przykładzie - 3 "tokeny", każdy o wymiarze 2.

In [ ]:
# 3 tokeny, każdy ma 2-wymiarowy embedding
x_demo = torch.tensor([
    [1.0, 0.0],   # token 0
    [0.0, 1.0],   # token 1
    [1.0, 1.0],   # token 2
])

# Naiwnie: dla każdej pozycji t, średnia z x_demo[:t+1]
wynik_petla = torch.zeros_like(x_demo)
for t in range(x_demo.shape[0]):
    wynik_petla[t] = x_demo[:t+1].mean(dim=0)

print("Średnia liczona pętlą:")
print(wynik_petla)

Pętla działa, ale jest wolna. Ten sam wynik można uzyskać **mnożeniem macierzy** - i to jest cała magia uwagi.

Tworzymy macierz `W` o kształcie (3, 3), w której wiersz `t` to wagi mówiące "ile wziąć z każdego z poprzednich tokenów". Dla zwykłej średniej kroczącej:

```
W = [[1.0, 0.0, 0.0],
     [0.5, 0.5, 0.0],
     [1/3, 1/3, 1/3]]
```

Zauważ, że `W` jest **dolnotrójkątna** - token na pozycji `t` nie może patrzeć w przyszłość! To kluczowe (inaczej model na treningu "oszukiwałby", widząc odpowiedź).

In [ ]:
# Budujemy macierz wag dolnotrójkątną z normalizacją wierszami
tril = torch.tril(torch.ones(3, 3))
W = tril / tril.sum(dim=1, keepdim=True)
print("Macierz wag W:")
print(W)

wynik_macierz = W @ x_demo
print("\nŚrednia liczona mnożeniem macierzy:")
print(wynik_macierz)

print("\nCzy zgadza się z pętlą?", torch.allclose(wynik_macierz, wynik_petla))

### 3. Od średniej do uwagi: niech każdy token sam zdecyduje, na co patrzy

Średnia ważona z równymi wagami nie jest mądra - traktuje wszystkie poprzednie tokeny tak samo. A gdyby tak każdy token mógł sam wybrać, na co chce patrzeć?

To właśnie robi **uwaga (attention)**. Każdy token tworzy:
- **query (Q)** - "czego szukam?" (np. "szukam czasownika")
- **key (K)** - "co ja oferuję?" (np. "jestem rzeczownikiem")
- **value (V)** - "co przekażę dalej, jeśli mnie wybiorą?"

Wagi obliczamy jako iloczyn skalarny `Q · K`: jeśli mój query pasuje do twojego key, dostajesz wysoką wagę. Potem softmax na każdym wierszu zamienia te "oceny" na prawdopodobieństwa, a maska trójkątna pilnuje, by nikt nie patrzył w przyszłość.

Wzór (z artykułu *Attention is all you need*):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

Dzielenie przez $\sqrt{d_k}$ jest po to, by softmax nie "saturował się" przy dużych wymiarach embeddingu. Zobaczmy to w kodzie.

In [ ]:
# Pokażmy uwagę krok po kroku na losowych danych
B, T, C = 2, 4, 8  # batch=2, czas=4 tokeny, embedding=8
x_przyklad = torch.randn(B, T, C)

head_size = 16  # Wymiar przestrzeni Q i K (może być inny niż C)

# Trzy warstwy liniowe - każda "projektuje" embedding na inną przestrzeń
W_query = nn.Linear(C, head_size, bias=False)
W_key   = nn.Linear(C, head_size, bias=False)
W_value = nn.Linear(C, head_size, bias=False)

Q = W_query(x_przyklad)  # (B, T, head_size)
K = W_key(x_przyklad)    # (B, T, head_size)
V = W_value(x_przyklad)  # (B, T, head_size)

# Wagi: dla każdej pary (i, j) liczymy Q_i · K_j
wagi = Q @ K.transpose(-2, -1) / (head_size ** 0.5)  # (B, T, T)

# Maska trójkątna - blokujemy patrzenie w przyszłość
tril = torch.tril(torch.ones(T, T))
wagi = wagi.masked_fill(tril == 0, float("-inf"))

# Softmax zamienia oceny na prawdopodobieństwa (sumują się do 1 w każdym wierszu)
wagi = F.softmax(wagi, dim=-1)

# I średnia ważona z wartości V
wynik = wagi @ V  # (B, T, head_size)

print("Wagi uwagi dla pierwszego elementu w batchu:")
print(wagi[0].round(decimals=2))
print(f"\nKształt wyjścia: {wynik.shape}")

Zwróć uwagę: każdy wiersz macierzy wag sumuje się do 1, a powyżej przekątnej są zera (maska). Wiersz `t` to "przepis", jak token na pozycji `t` miesza wartości `V` z poprzednich pozycji.

**Najważniejsze**: wagi są **danymi-zależne**! Inny tekst → inne Q i K → inne wagi. Sieć sama uczy się, na co patrzeć.

### 4. Klasa `Head` - jedna głowa uwagi jako moduł

Pakujemy powyższy kod w przyzwoity moduł PyTorch. Maskę trójkątną rejestrujemy jako `buffer` - to znaczy, że nie jest to parametr trenowalny, ale podróżuje z modelem (np. na GPU).

In [ ]:
class Head(nn.Module):
    """Jedna głowa samouwagi (causal self-attention)."""
    def __init__(self, embedding_dim: int, head_size: int, block_size: int):
        super().__init__()
        self.key   = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        # Maska trójkątna - rejestrujemy ją jako bufor (nie jest parametrem)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # Liczymy wagi uwagi
        wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)  # (B, T, T)
        wagi = wagi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wagi = F.softmax(wagi, dim=-1)

        return wagi @ v  # (B, T, head_size)

### 5. Mini-model z jedną głową uwagi

Składamy całość:
1. **Embedding tokenów** - każdy znak → wektor.
2. **Embedding pozycji** - dodatkowo każda pozycja (0, 1, ..., block_size-1) ma swój wektor. Bez tego model nie wiedziałby, w jakiej kolejności są tokeny - sama uwaga jest "bez kolejności".
3. **Jedna głowa uwagi** miesza informacje między pozycjami.
4. **Warstwa wyjściowa** - dla każdej pozycji oddzielnie wypluwa logity (oceny) dla każdego znaku w słowniku.

In [ ]:
class JednoglowyModel(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, head_size: int, block_size: int):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, embedding_dim)
        self.pos_emb = nn.Embedding(block_size, embedding_dim)
        self.head = Head(embedding_dim, head_size, block_size)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        tok = self.token_emb(idx)                                 # (B, T, embedding_dim)
        pos = self.pos_emb(torch.arange(T, device=idx.device))    # (T, embedding_dim)
        x = tok + pos                                              # (B, T, embedding_dim)
        x = self.head(x)                                           # (B, T, head_size)
        logits = self.lm_head(x)                                   # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generuj(self, idx: torch.Tensor, max_nowych: int, temperatura: float = 1.0) -> torch.Tensor:
        """Generuje max_nowych tokenów dopisanych do idx (B, T)."""
        for _ in range(max_nowych):
            # Bierzemy ostatnie block_size tokenów (model nie widzi dalej)
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)              # (B, T, vocab_size)
            logits = logits[:, -1, :] / temperatura  # bierzemy ostatnią pozycję
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
embedding_dim = 32
head_size = 32

model = JednoglowyModel(vocab_size, embedding_dim, head_size, block_size)

# Liczba parametrów
n_params = sum(p.numel() for p in model.parameters())
print(f"Liczba parametrów: {n_params:,}")
print(model)

### 6. Trening

Funkcja straty: cross-entropy. Subtelność: wyjście modelu ma kształt `(B, T, vocab_size)`, a cele `(B, T)`. PyTorch wymaga "spłaszczenia" do `(B*T, vocab_size)` i `(B*T,)`. Dzięki temu uczymy się przewidywać następny znak na **każdej** pozycji w sekwencji jednocześnie.

In [ ]:
def oblicz_strate(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    logits = model(x)                              # (B, T, vocab_size)
    B, T, V = logits.shape
    return F.cross_entropy(logits.view(B*T, V), y.view(B*T))

@torch.no_grad()
def oszacuj_strate_test(model: nn.Module, n_paczek: int = 20) -> float:
    model.eval()
    straty = []
    for _ in range(n_paczek):
        xb, yb = losuj_paczke("test")
        straty.append(oblicz_strate(model, xb, yb).item())
    model.train()
    return sum(straty) / len(straty)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
epochs = 2000
eval_every = 50

liveloss = PlotLosses()
model.train()

for epoch in range(epochs):
    xb, yb = losuj_paczke("train")
    loss = oblicz_strate(model, xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % eval_every == 0 or epoch == epochs - 1:
        liveloss.update({"loss": loss.item(), "val_loss": oszacuj_strate_test(model)})
        liveloss.send()

print("Trening zakończony.")

### 7. Generowanie tekstu

Zaczynamy od dowolnego ciągu (np. `"Litwo"`), zamieniamy go na indeksy i pozwalamy modelowi dopisywać kolejne znaki.

In [ ]:
def generuj_tekst(model: nn.Module, start_str: str, dlugosc: int = 300, temperatura: float = 1.0) -> str:
    idx = torch.tensor([[char2id[c] for c in start_str]], dtype=torch.long)
    out = model.generuj(idx, max_nowych=dlugosc, temperatura=temperatura)
    return "".join(id2char[i.item()] for i in out[0])

print(generuj_tekst(model, "Litwo, ojczyzno moja", dlugosc=400, temperatura=0.8))

### 8. Co dalej?

Mamy działającą jedną głowę uwagi. Już samo to powinno dać tekst lepszy niż MLP z Notebooka 3 - dłuższy kontekst i lepsze "słuchanie" odpowiednich tokenów.

Ale prawdziwy GPT to:
- **Wiele głów uwagi równolegle** (multi-head attention) - każda głowa może zwracać uwagę na coś innego (jedna na samogłoski, druga na końcówki słów, ...).
- **Warstwy feed-forward** po każdej uwadze - moment na "przemyślenie" zebranej informacji.
- **Stos kilku takich bloków** - każda warstwa coraz bardziej abstrakcyjne wzorce.
- **Połączenia rezydualne i LayerNorm** - bez nich głębokie sieci się nie trenują.

Tym wszystkim zajmiemy się w [Notebooku 5](5_mini_gpt.ipynb).